In [3]:
import time
import numpy as np
import random
import os
import copy
import math

import pandas as pd
from sklearn import preprocessing
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import minmax_scale
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
#from xgboost import XGBClassifier
from scipy.special import softmax
import scipy.stats as ss

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns;
# sns.set_theme(color_codes=True)
import warnings


In [4]:
warnings.filterwarnings("ignore")

random_seed = 0

torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(random_seed)
random.seed(random_seed)

#torch.cuda.set_device("cuda:0")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['CUDA_VISIBLE_DEVICES'] = '1'


In [3]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0.0001, path='checkpoint_multi.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.

            verbose (bool): If True, prints a message for each validation loss improvement.

            delta (float): Minimum change in the monitored quantity to qualify as an improvement.

            path (str): Path for the checkpoint to be saved to.

            trace_func (function): trace print function.

        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter > self.patience:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(
                f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 定义用户提供的DNN模型
class DNN(nn.Module):
    def __init__(self, In_Nodes, Modules):
        super(DNN, self).__init__()
        self.Modules = Modules
        self.task_FC1_x = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC1_y = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC2 = nn.Sequential(nn.Linear(Modules*2, 32), nn.ReLU())
        self.task_FC3 = nn.Sequential(nn.Linear(32, 16), nn.ReLU())
        self.task_FC4 = nn.Linear(16, 5)  # 输出logits

    def forward(self, xg):
        xg_x = self.task_FC1_x(xg)
        xg_y = self.task_FC1_y(xg)
        xg = torch.cat([xg_x.reshape(-1, 1, self.Modules), xg_y.reshape(-1, 1, self.Modules)], dim=1)
        norm = torch.norm(xg, dim=1, keepdim=True)
        xg = xg.div(norm + 1e-8)  # 防止除零
        xg = xg.view(-1, self.Modules * 2)
        xg = self.task_FC2(xg)
        xg = self.task_FC3(xg)
        xg = self.task_FC4(xg)
        return xg

def cal_label(test):
    array = []
    for i in test:
        temp = np.max(i)  # 找到当前行的最大值
        count = 0  # 初始化计数器为0
        for j in i:
            if temp == j: 
                break
            count += 1
        array.append(count)  # 将计数器的值添加到结果列表中
    return np.array(array)  # 将结果列表转换为NumPy数组并返回


In [ ]:

# import data
mrna = pd.read_csv(
    'Datasets\mRNA.txt', sep='\t')
meth = pd.read_csv(
    'Datasets\DNA methylation.txt', sep='\t')
mcnv = pd.read_csv(
    'Datasets\CNV.txt', sep='\t')
clincal = pd.read_csv(
    'Datasets\label.txt', sep='\t')

mrna.index = mrna['gene_name']
del mrna['gene_name']

meth.index = meth['gene_name']
del meth['gene_name']

mcnv.index = mcnv['gene_name']
del mcnv['gene_name']

clincal.index = clincal['id']
del clincal['id']

label = clincal['subtype']


mrna=mrna.T
mrna = mrna.iloc[:, :]
meth=meth.T
meth = meth.iloc[:, :]
mcnv=mcnv.T
mcnv=mcnv.iloc[:,:]

mrna = mrna.values
meth = meth.values
mcnv=mcnv.values

min_max_scaler = preprocessing.MinMaxScaler()
mrna = min_max_scaler.fit_transform(mrna)
meth = min_max_scaler.fit_transform(meth)
mcnv = min_max_scaler.fit_transform(mcnv)
label_num, uniques = pd.factorize(label)
label = label_num

stacked_features = torch.cat((mrna,meth,mcnv), dim=1).to(device)
# Split data
Xg_train, Xg_test, yg_train, yg_test = train_test_split(stacked_features, label, test_size=0.2, random_state=42)


Xg_train = Xg_train.astype(float)
Xg_test = Xg_test.astype(float)

y_train = np.array(yg_train).flatten().astype(int)
y_test = np.array(yg_test).flatten().astype(int)

Xg = torch.tensor(Xg_train, dtype=torch.float32).to(device)


Xg_test = torch.tensor(Xg_test, dtype=torch.float32).to(device)

y = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

'''
ds = TensorDataset(Xg, Xm,Xc, y)
loader = DataLoader(ds, batch_size=y_train.shape[0], shuffle=True)

In_Nodes1 = Xg_train.shape[1]
In_Nodes2 = Xm_train.shape[1]
In_Nodes3 = Xc_train.shape[1]'''


In [ ]:

class BoostedDNN:
    def __init__(self, base_model_class, n_estimators=5, lr=0.001, modules=64, max_epochs=100,path=''):
        self.base_model_class = base_model_class  # 基础模型类（用户提供的DNN）
        self.n_estimators = n_estimators        # 弱学习器数量
        self.lr = lr                            # 学习率
        self.modules = modules                  # 基础模型参数
        self.max_epochs = max_epochs            # 单模型训练轮次
        self.models = []                        # 存储所有弱学习器
        self.weights = []                       # 各模型的集成权重
        self.path=path

    def _calculate_residual(self, y_true, y_pred_logits):
        """计算伪残差（交叉熵的负梯度）"""
        probs = torch.softmax(y_pred_logits, dim=1)
        one_hot = torch.eye(5)[y_true.long()].to(y_pred_logits.device)
        return one_hot - probs  # 梯度公式: grad = pred_prob - true_one_hot

    def fit(self, X, y, in_nodes):
        """训练Boosting模型"""
        # 初始化原始预测和残差
        device = X.device if isinstance(X, torch.Tensor) else 'cpu'
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        y = torch.as_tensor(y, dtype=torch.long, device=device)
        y_onehot = torch.eye(5)[y].float().to(device)
        residual = y_onehot.clone()
        
        for i in range(self.n_estimators):
            # 创建并训练当前弱学习器
            model = self.base_model_class(In_Nodes=in_nodes, Modules=self.modules).to(device)
            optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=1e-4)  # 添加L2正则
            early_stopping = EarlyStopping(patience=20, verbose=False,path=self.path)
            # 训练当前模型拟合残差
            dataset = TensorDataset(X, residual)
            loader = DataLoader(dataset, batch_size=64, shuffle=True)
            
            for epoch in range(self.max_epochs):
                model.train()
                total_loss = 0.0
                for batch_X, batch_residual in loader:
                    optimizer.zero_grad()
                    pred_logits = model(batch_X)
                    loss = nn.MSELoss()(pred_logits, batch_residual)  # 用MSE拟合残差
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
                    optimizer.step()
                    total_loss += loss.item()
                early_stopping(total_loss, model)
                if early_stopping.early_stop:
                    #print("Early stopping")
                    #print("--------------------------------------------------------------------------------------------------")
                    break
                '''
                if (epoch + 1) % 1000 == 0 or epoch == 0:
                    print(f"Model {i+1} Epoch [{epoch+1}/{self.max_epochs}], Loss: {total_loss/len(loader):.4f}")
                '''
            # 计算动态权重并更新残差
            with torch.no_grad():
                model.eval()
                pred_logits = model(X)
                residual_current = self._calculate_residual(y, pred_logits)
                weight = 1.0 / (torch.norm(residual_current).item() + 1e-8)  # 基于残差范数分配权重
                self.weights.append(weight)
                self.models.append(model)
                residual = residual_current  # 更新残差

    def predict(self, X):
        """集成预测（加权投票）"""
        device = next(self.models[0].parameters()).device  # 获取模型设备

        X = torch.as_tensor(X, dtype=torch.float32, device='cpu')
        final_logits = torch.zeros(len(X), 5, device=device)
        
        for model, weight in zip(self.models, self.weights):
            with torch.no_grad():
                logits = model(X)
                final_logits += weight * logits  # 加权累加logits
    
        return final_logits


In [ ]:
#DNA
start = time.time()

# 初始化并训练Boosting模型
DNA_model = BoostedDNN(
    base_model_class=DNN,
    n_estimators=10,
    lr=0.0001,
    modules=128,
    max_epochs=30000,
    path='model\dna.pt'
    )
DNA_model.fit(Xg_train, y_train, Xg_train.shape[1])

# 预测并评估
y_pred = DNA_model.predict(Xg_test)
accuracy = accuracy_score(y_test.cpu().numpy(), torch.argmax(y_pred, dim=1).cpu().numpy())
print(f"Boosting DNN准确率: {accuracy:.4f}")
print("time :", time.time() - start)
